<a href="https://colab.research.google.com/github/LuciaKajanova/dspracticum25_flowers_team/blob/martin_upravy/gemma3_finetune_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning GEMMY3 na datech
Co se povedlo:

Technicky je proces fine-tuningu v pořáadku, model se naučil dodržovat tvůj specifický výstupní formát.

Co se nepovedlo (nebo spíše nemělo velký dopad):

Vzhledem k malému datasetu (100 položek) a vysoké kvalitě základního modelu nepřinesl fine-tuning výrazné zlepšení v kvalitě samotných odpovědí oproti netrénované verzi, viz dolní část skriptu.

In [1]:
############## KONFIGURACE A INSTALACE ##############

# Konfigurace souborů (Musí být nahrány do Colab adresáře)
TRAIN_FILE = "train_100.jsonl"
TEST_FILE = "test_99.jsonl"
OUTPUT_DIR = "gemma_3_4b_pisne_lora" # Adresář pro uložení modelu

# Instalace Unsloth a potřebných knihoven
# Používáme stabilní a rychlou instalaci pro Colab
!pip install -q --upgrade "unsloth[colab-new]>=2024.7" torch trl datasets accelerate peft bitsandbytes

import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import json
import os
import random

# --- Globální konfigurace modelu ---
max_seq_length = 2048
dtype = None # None pro auto detekci (bfloat16 na A100/H100, float16 na T4/V100)
load_in_4bit = True # QLoRA pro úsporu VRAM



 ############## 2. NAČTENÍ MODELU A LoRA KONFIGURACE ##############

# načtení modelu
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it", #výběr modelu
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# aplikace LoRA adaptérů
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,             # rank matice
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,    # škálovací faktor
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Optimalizace VRAM
    random_state = 3407,
)


 ##############  3. PŘÍPRAVA VLASTNÍCH DAT (Nahrazení Alpaca datasetu)  ##############

print("\n3. Načítám a formátuji TVÉ datové sady...")

# Kontrola souborů a načtení
if not (os.path.exists(TRAIN_FILE) and os.path.exists(TEST_FILE)):
    print(f"KRITICKÁ CHYBA: Soubory {TRAIN_FILE} a {TEST_FILE} nebyly nalezeny.")
    raise FileNotFoundError("Chybí vstupní JSONL soubory. Nahraj je do Colab adresáře a spusť znovu.")

datasets = load_dataset("json", data_files={"train": TRAIN_FILE, "test": TEST_FILE})

# Definice formátu, na kterém model trénoval (Alpaca-like)
# POZOR: POUŽÍVÁME ZDE TŘI ZNAKY ### JAKO V UNLSOTH ŠABLONĚ
ALPACA_PROMPT = """### Instruction:{}### Input:{}### Response:{}"""
EOS_TOKEN = tokenizer.eos_token # Token pro konec sekvence

def formatting_prompts_func(examples):
    texts = []
    # Zde procházíme sloupce z nahrnaých  JSONL souboru
    for instruction, input_text, output_text in zip(examples["instruction"], examples["input"], examples["output"]):
        # Musíme přidat EOS_TOKEN, jinak model generuje donekonečna!
        text = ALPACA_PROMPT.format(instruction, input_text, output_text) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Aplikace formátování
datasets = datasets.map(formatting_prompts_func, batched = True,)

print(f"Data načtena a naformátována. Train: {datasets['train'].num_rows} ks, Test: {datasets['test'].num_rows} ks.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients

3. Načítám a formátuji TVÉ datové sady...
Data načtena a naformátována. Train: 100 ks, Test: 99 ks.


trenink (riziko overfittingu ale zdalo se jako nutnost pri menší teprature a malemu mnotsvi finetunovacích dat) bere se však best state model tedy vyberes dohormady njelepší stav modelu po všech epochách

In [2]:
 ############## 4. TRÉNOVÁNÍ MODELU ##############

if datasets["train"].num_rows > 0:
    # Zde definujeme, že se použije SFTTrainer
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = datasets["train"],
        eval_dataset = datasets["test"],
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            num_train_epochs = 10,  # zvýšeno na 10 epoch
            learning_rate = 2e-5,   #     snizeno pro stabilnější konvergenci
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 1,
            optim = "adamw_8bit",
            weight_decay = 0.05,    #  zvýšeno proti overfittingu
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = OUTPUT_DIR,
            eval_strategy = "epoch",
            save_strategy = "epoch",
            load_best_model_at_end = True,
            report_to = "none",
        ),
    )

    trainer_stats = trainer.train()

    # Uložení finálního modelu (vynucené 4bit uložení, protože je to finální krok)
    model.save_pretrained_merged(OUTPUT_DIR + "_merged_10ep", tokenizer, save_method = "merged_4bit_forced")
    print(f"\n Trénink dokončen! Finální model uložen jako '{OUTPUT_DIR}_merged_10ep'.")

Unsloth: Switching to float32 training since model cannot work with float16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 10 | Total steps = 130
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,2.033500,3.474569
2,1.599000,3.087332
3,1.449900,2.878071
4,1.441600,2.708075
5,1.327900,2.601478
6,0.944100,2.555156
7,1.164000,2.533703
8,1.217800,2.521096
9,1.034000,2.515813
10,0.956300,2.514202


Unsloth: Not an error, but Gemma3ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Merging LoRA weights into 4bit model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Unsloth: Merging finished.
Unsloth: Found skipped modules: ['model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj', 'model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj', 'model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj', 'model.vision_tower.vision_model.encoder.layers.0.self_attn.out_proj', 'model.vision_tower.vision_model.encoder.layers.0.mlp.fc1', 'model.vision_tower.vision_model.encoder.layers.0.mlp.fc2', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.k_proj', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.v_proj', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.q_proj', 'model.vision_tower.vision_model.encoder.layers.1.self_attn.out_proj', 'model.vision_tower.vision_model.encoder.layers.1.mlp.fc1', 'model.vision_tower.vision_model.encoder.layers.1.mlp.fc2', 'model.vision_tower.vision_model.encoder.layers.2.self_attn.k_proj', 'model.vision_tower.vision_model.encoder.layers.2.self_attn.v_proj', 'model

inference a test
(zde byl problém že model stejně generoval  výstup v anglicitne ze finetunig na tak malých datech nepomohl model přeučit na odpovedi v čestine, vynutili sme si tak at odpovida v cestine
) je potřeba dávat pozor na temprature!!


In [8]:
import torch
import gc
import os
from unsloth import FastLanguageModel

# --- POMOCNÁ FUNKCE PRO ČIŠTĚNÍ VRAM ---
def vycistit_vram():
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM vyčištěna.")
# 1. KONFIGURACE A DATA

ULOZENY_MODEL_DIR = "gemma_3_4b_pisne_lora_merged_10ep"

INTERPRET = "Karel Kryl"
NAZEV_SKLADBY = "Král a klaun"
TEXT_SKLADBY = """
Král do boje táh
Do veliké dálky
A s ním do té války
Jel na mezku klaun
Než hledí si stáh
Tak z výrazu tváře
Bys nepoznal lháře
Co zakrývá strach
Tiše šeptal při té hrůze
Inter arma silent Musae
Místo zvonku cinkal brněním
Král do boje táh
Do veliké dálky
A s ním do té války
Jel na mezku klaun
Král do boje táh
A sotva se vzdálil
Tak vesnice pálil
A dobýval měst
Klaun v očích měl hněv
Když sledoval žháře
Jak smývali v páře
Prach z rukou a krev
Tiše šeptal při té hrůze
Inter arma silent Musae
Místo loutny držel v ruce meč
Král do boje táh
A sotva se vzdálil
Tak vesnice pálil
A dobýval měst
Král do boje táh
S tou vraždící lůzou
Klaun třásl se hrůzou
A odvetu kul
Když v noci byl klid
Tak oklamal stráže
A nemaje páže
Sám burcoval lid
Všude křičel do té hrůzy
Ve válce že mlčí Múzy
Muži by však mlčet neměli
Král do boje táh
S tou vraždící lůzou
Klaun třásl se hrůzou
A odvetu kul
Král do boje táh
A v červáncích vlídných
Zřel na čele bídných
Jak vstříc jde mu klaun
Když západ pak vzplál
Tok potoků temněl
Klaun tušení neměl
Jak zahynul král
Kdekdo křičel při té hrůze
Inter arma silent Musae
Krále z toho strachu trefil šlak
Klaun tiše se smál
A zem žila dále
A neměla krále
Klaun na loutnu hrál
Klaun na loutnu hrál
Klaun na loutnu hrál
"""
MODEL_VSTUP = f"Interpret: {INTERPRET}\nNázev: {NAZEV_SKLADBY}\nText:\n{TEXT_SKLADBY}"
CZECH_HINT = "Alt. název: " # Tvůj osvědčený hint pro vynucení formátu

# 2. UNIVERZÁLNÍ TESTOVACÍ FUNKCE

def rychly_test(nazev_testu, model_obj, tok_obj, instr="", vstup="", hint="", temp=0.4):
    print(f"\n--- {nazev_testu} ---")
    FastLanguageModel.for_inference(model_obj)
    # Sestavení promptu
    base_prompt = f"### Instruction:{instr}\n### Input:{vstup}\n### Response:{hint}" if instr \
             else f"### Input:{vstup}\n### Response:{hint}"
    # Generování
    inputs = tok_obj(text=[base_prompt], return_tensors="pt").to("cuda")
    outputs = model_obj.generate(**inputs, max_new_tokens=128, temperature=temp, use_cache=True,
                                 eos_token_id=tok_obj.eos_token_id, pad_token_id=tok_obj.pad_token_id)
    # Dekódování a získání čisté odpovědi
    full_output = tok_obj.decode(outputs[0], skip_special_tokens=True)
    response_only = full_output.split("### Response:")[-1]
    print(response_only.strip())

# 3. SPUŠTĚNÍ TESTŮ
# Příklad 1: AGRESIVNÍ TEST (Bez instrukce, ale s HINTEM)
rychly_test("AGRESIVNÍ TEST (s hintem)", model, tokenizer,
            vstup=MODEL_VSTUP,
            instr="",
            hint=CZECH_HINT) # <--- Zde předáváme "Alt. název: "

# Příklad 2: STANDARDNÍ INSTRUKCE (s hintem)
rychly_test("STANDARDNÍ INSTRUKCE (s hintem)", model, tokenizer,
            vstup=MODEL_VSTUP,
            instr="Analyzuj píseň, vytvoř alternativní název, shrnutí (1 větu) a ohodnoť náladu (1-5).",
            hint=CZECH_HINT)

# Příklad 3: Jiná instrukce (bez hintu, pro srovnání)
rychly_test("DOTAZ NA POSTAVU (bez hintu)", model, tokenizer,
            vstup=MODEL_VSTUP,
            instr="Kdo je hlavní postavou a co jí hrozí?")
# BASELINE MODEL výstup
if True:
    if 'model' in globals(): del model, tokenizer
    vycistit_vram()

    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/gemma-3-4b-it",
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )
    rychly_test("[B] BASELINE: STEJNÁ INSTRUKCE", base_model, base_tokenizer, vstup=MODEL_VSTUP,
                instr="Analyzuj píseň, vytvoř alternativní název, shrnutí (1 větu) a ohodnoť náladu (1-5).")

# ZNOVUNAČTENÍ FINE-TUNED MODELU

print(f"\nPŘEPÍNÁM ZPĚT NA FT MODEL '{ULOZENY_MODEL_DIR}'...")
if 'base_model' in globals(): del base_model, base_tokenizer
vycistit_vram()

if os.path.exists(ULOZENY_MODEL_DIR):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = ULOZENY_MODEL_DIR,
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )
    print("FT model načten.")

else:
    print(f"CHYBA: Složka '{ULOZENY_MODEL_DIR}' neexistuje.")


--- AGRESIVNÍ TEST (s hintem) ---
Alt. název: 
Interpretace:
Karel Kryl v básni "Král a klaun" zobrazuje tragédii války a její dopad na lidské duše. Báseň se odehrává v malé vesnici, která je napadena a ničená vojskem. Král, který se do války vrhá, je zbytečný a neúspěšný. Klaun, který je svědkem hrůz války, se snaží o protest a odvetu. Báseň je plná symboliky a metafor, jako je "Inter arma silent Musae" (

--- STANDARDNÍ INSTRUKCE (s hintem) ---
Alt. název: 
Válka a loutník
Shrnutí: Píseň popisuje krále, který se vydává do války, zatímco klaun se snaží skrýt svůj strach a nakonec se stává katalyzátorem odporu proti válce.
Nálada: 3

--- DOTAZ NA POSTAVU (bez hintu) ---
Hlavní postavou je Karel Kryl (jako interpret) a co mu hrozí, je válka a zkáza, které se projevují v podobě plamenů, vraždění a ztráty krále.
VRAM vyčištěna.
==((====))==  Unsloth 2025.11.2: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_